# 02 — QLoRA Experiment

Fine-tune Gemma-3-270m with 4-bit quantization + LoRA adapters.
Compare VRAM usage and accuracy against baseline.

In [ ]:
import sys
sys.path.insert(0, '../src')

from transformers import AutoTokenizer
from llm_optimization.core import load_config
from llm_optimization.data import load_and_prepare_data, QADataset
from llm_optimization.training import build_qlora_trainer
from llm_optimization.utils import ResourceMonitor

In [ ]:
config = load_config('../configs/qlora.yaml')
train_df, _, val_df = load_and_prepare_data(config.data)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token

train_ds = QADataset(train_df, tokenizer, config.data.max_length)
val_ds = QADataset(val_df, tokenizer, config.data.max_length)

In [ ]:
trainer, model = build_qlora_trainer(config, train_ds, val_ds, tokenizer)
trainer.train()

trainer.save_model()
tokenizer.save_pretrained(config.output_path)